# Пайплайн: A-Vision → список вещей → CLIP по каталогу → top-5 + URL

**Нужно:** GPU (T4), папка проекта с `fashion_search_bus.py`, `multimodal_search.py`, файл индекса `clip_index.pt`, картинки каталога.

**Опционально:** `metadata_final.json` от скрапера (поля `url`, `local_image_path`) и корень картинок — тогда в ответе будут ссылки на объявления.

1. Укажи `PROJECT_ROOT` и пути к данным ниже.  
2. Выполни ячейки по порядку.  
3. Сначала поднимается **шина (CLIP + индекс)**, затем **A-Vision** — так проще укладываться в VRAM.
4. Список вещей запрашивается **на английском** (по умолчанию), чтобы **CLIP** лучше искал по тем же фразам.

Если проект на **Google Drive** — выполни следующую ячейку (`drive.mount`), затем пути к `PROJECT_ROOT`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import sys

# --- ИЗМЕНИ под свой Drive / сессию Colab ---
PROJECT_ROOT = Path("/content/drive/MyDrive/path/to/2_photo_scrapper")

# Где лежит индекс CLIP (после multimodal_search --build-index)
DATA_ROOT = PROJECT_ROOT / "classifier_data"
CLIP_INDEX = DATA_ROOT / "clip_index.pt"

# metadata_final.json от avito_scraper + папка images (для URL)
METADATA_JSON = PROJECT_ROOT / "avito_data_fashion" / "metadata_final.json"
IMAGES_ROOT = PROJECT_ROOT / "avito_data_fashion" / "images"

if not PROJECT_ROOT.is_dir():
    raise SystemExit(f"Нет папки проекта: {PROJECT_ROOT} — смонтируй Drive и поправь путь.")
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CLIP_INDEX exists:", CLIP_INDEX.is_file())
print("METADATA_JSON exists:", METADATA_JSON.is_file())

In [ ]:
# Зависимости из requirements-colab.txt (без -U). Сначала ячейка с PROJECT_ROOT.
import subprocess
import sys
from pathlib import Path

_req = PROJECT_ROOT / "requirements-colab.txt"
if not _req.is_file():
    raise SystemExit(
        f"Нет файла {_req} — положи requirements-colab.txt в корень проекта на Drive."
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)],
    check=True,
)
print("Colab deps OK:", _req)


In [ ]:
from fashion_search_bus import FashionSearchBus, OUTFIT_EXTRACTION_PROMPT

meta = METADATA_JSON if METADATA_JSON.is_file() else None
img_root = IMAGES_ROOT if IMAGES_ROOT.is_dir() else None

bus = FashionSearchBus(
    index_path=CLIP_INDEX,
    metadata_path=meta,
    images_root=img_root,
    catalog_root=PROJECT_ROOT,
)
print("Шина: CLIP + индекс, записей:", len(bus.index_paths), "URL в карте:", len(bus.path_to_url))

## A-Vision (как в `COLAB_AVISION_last`)
Параметры и загрузка модели + `avision_ask`.

In [ ]:
import gc
import os
import re
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

assert torch.cuda.is_available(), "Включи GPU"

MODEL_ID = "AvitoTech/avision"
MAX_NEW_TOKENS = 384
USE_4BIT = True
MIN_PIXELS = 4 * 28 * 28
MAX_PIXELS = 768 * 28 * 28

torch.cuda.empty_cache()
gc.collect()

try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception:
    pass
_hf_kw = {}
if os.environ.get("HF_TOKEN"):
    _hf_kw["token"] = os.environ["HF_TOKEN"]

print(f"Загрузка {MODEL_ID!r} (4-bit={USE_4BIT})…")
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto", **_hf_kw
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, torch_dtype="auto", device_map="auto", **_hf_kw
    )
processor = AutoProcessor.from_pretrained(MODEL_ID, **_hf_kw)


def clean_avision_output(text: str) -> str:
    if not text:
        return text
    t = text.replace("<0x0A>", "\n").replace("<0x0a>", "\n")
    t = t.replace("\u2581", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


def avision_ask(prompt, image, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS
    if isinstance(image, (str, Path)):
        pil = Image.open(image).convert("RGB")
    else:
        pil = image.convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": pil,
                    "min_pixels": MIN_PIXELS,
                    "max_pixels": MAX_PIXELS,
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]
    chat_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[chat_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    in_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[0, in_len:]
    tok = getattr(processor, "tokenizer", None) or processor
    raw = tok.decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
    return clean_avision_output(raw)

print("A-Vision готов.")

## Фото человека и запуск шины

In [ ]:
from pathlib import Path

USE_UPLOAD = True
if USE_UPLOAD:
    from google.colab import files
    up = files.upload()
    assert up, "Загрузи изображение"
    IMAGE_PATH = Path(list(up.keys())[0])
else:
    IMAGE_PATH = Path("/content/sample.jpg")

print("IMAGE_PATH:", IMAGE_PATH.resolve())

In [ ]:
import json

from fashion_search_bus import display_outfit_pipeline_images

result = bus.run_outfit_pipeline(
    IMAGE_PATH,
    avision_ask,
    outfit_prompt=OUTFIT_EXTRACTION_PROMPT,
    top_k_per_phrase=5,
    fetch_k=40,
)

print("=== Сырой ответ VLM ===")
print(result.raw_avision)
print("\n=== Распарсенные фразы ===")
print(result.items)
print("\n=== JSON (для API / сохранения) ===")
print(json.dumps(result.to_dict(), ensure_ascii=False, indent=2))

# Превью картинок + ссылки (Colab / Jupyter). В обычном терминале — только лог.
display_outfit_pipeline_images(result, project_root=PROJECT_ROOT, max_per_phrase=5)